In [1]:
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_dataset.csv")

In [3]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,male,blouse,clothing,53,kentucky,l,gray,winter,3.1,yes,express,yes,yes,14,venmo,fortnightly
1,2,19,male,sweater,clothing,64,maine,l,maroon,winter,3.1,yes,express,yes,yes,2,cash,fortnightly
2,3,50,male,jeans,clothing,73,massachusetts,s,maroon,spring,3.1,yes,free shipping,yes,yes,23,credit card,weekly
3,4,21,male,sandals,footwear,90,rhode island,m,maroon,spring,3.5,yes,next day air,yes,yes,49,paypal,weekly
4,5,45,male,blouse,clothing,49,oregon,m,turquoise,spring,2.7,yes,free shipping,yes,yes,31,paypal,annually


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3900 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

## A : Behavirol Features
* frequency_score
* subscription_flag
* repeat_buyer_flag

### 1. frequency_score

This feature converts catagorical purchase frequency into a numerical behavioral score

Customer with high score (purchasing more frequently) demonstrate:
- higher engagement
- stronger retention probability
- long term value potential

In [5]:
freqency_map = {
    'weekly' : 7,
    'bi-weekly' : 6,
    'fortnightly' : 5,
    'monthly' : 4,
    'quarterly' : 3,
    'every 3 months' : 2,
    'annually' : 1
}

df["frequency_score"] = df["Frequency of Purchases"].map(freqency_map)

In [7]:
df[["Frequency of Purchases", "frequency_score"]].head()

,Frequency of Purchases,frequency_score
0,fortnightly,5
1,fortnightly,5
2,weekly,7
3,weekly,7
4,annually,1


### 2. subscription_flag

This features tells whether customer is subscribed to brand's membership (binary : 0, 1)

Subscribed customer demonstrate:
- more frequent engagement
- more stronger retention
- higher lifetime value

In [19]:
df["subscription_flag"] = (df["Subscription Status"] == "yes").astype(int)

In [20]:
df[["Subscription Status", "subscription_flag"]].head()

,Subscription Status,subscription_flag
0,yes,1
1,yes,1
2,yes,1
3,yes,1
4,yes,1


### 3. repeat_buyer_flag

Tells if a customer has demonstrated repete purchasing behaviour 

More than 6 purchases  == repeated purchasing behavoiur

binary : 0, 1

In [81]:
df["repeat_buyer_flag"] = (df["Previous Purchases"] > 6).astype(int)

In [82]:
df[["Previous Purchases", "repeat_buyer_flag"]].head()

,Previous Purchases,repeat_buyer_flag
0,14,1
1,2,0
2,23,1
3,49,1
4,31,1


In [83]:
df[
    [

        "repeat_buyer_flag"
       
    ]
].describe()

,repeat_buyer_flag
count,3900.000000
mean,0.868974
std,0.337472
min,0.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,1.000000


### 4. satisfaction_flag

This feature identifes highly satisfied customers using review rating

Customer with review rating >= 4 means highly satisfied

binary : 0, 1

In [84]:
df["satisfaction_flag"] = (df["Review Rating"] >= 4).astype(int)

In [85]:
df[["Review Rating", "satisfaction_flag"]].head()

,Review Rating,satisfaction_flag
0,3.1,0
1,3.1,0
2,3.1,0
3,3.5,0
4,2.7,0


In [86]:
df[
    [
        "frequency_score",
        "subscription_flag",
        "repeat_buyer_flag",
        "satisfaction_flag"
    ]
].describe()

,frequency_score,subscription_flag,repeat_buyer_flag,satisfaction_flag
count,3900.000000,3900.000000,3900.000000,3900.000000
mean,3.950256,0.270000,0.868974,0.418974
std,2.001433,0.444016,0.337472,0.493454
min,1.000000,0.000000,0.000000,0.000000
25%,2.000000,0.000000,1.000000,0.000000
50%,4.000000,0.000000,1.000000,0.000000
75%,6.000000,1.000000,1.000000,1.000000
max,7.000000,1.000000,1.000000,1.000000


## B : Promo Sesitive Features
* promo_dependency_score
* discount_sensitivity_segment

### 1. promo_dependency_score

This shows how dependent a customer is on promo codes and dicounts

It is inferred using 'Discount Applied' and 'Promo Code Used'

Higher score means highly dependent on pormo codes and dicounts

(discount applied) + (promo code used) = score
- yes + yes = 1
- yes + no = 0.5
- no + yes = 0.5
- no + no = 0

In [87]:
def promo_dependency(row):

    if (
        row["Discount Applied"] == "yes" and
        row["Promo Code Used"] == "yes"
    ):
        return 1.0

    elif (
        row["Discount Applied"] == "yes"
    ):
        return 0.5

    else:
        return 0.0

In [88]:
df["promo_dependency_score"] = (df.apply(promo_dependency, axis = 1))

In [89]:
df[
    [
        "Discount Applied",
        "Promo Code Used",
        "promo_dependency_score"
    ]
].head()

,Discount Applied,Promo Code Used,promo_dependency_score
0,yes,yes,1.0
1,yes,yes,1.0
2,yes,yes,1.0
3,yes,yes,1.0
4,yes,yes,1.0


### 2. discount_sensitivity_segment

This feature converts promo_dependency_score into businedd friendly customer segments

These segments help identify:
- organic buyers
- moderatley discount sensitive customers
- highly promo dependent customers

In [90]:
def discount_segment(score):
    if score == 0:
        return "organic"

    elif score == 0.5:
        return "moderate"

    else:
        return "highly promo sensitive"

In [91]:
df["discount_sensitivity_segment"] = df['promo_dependency_score'].apply(discount_segment)   

In [92]:
df[
    [
        "promo_dependency_score",
        "discount_sensitivity_segment"
    ]
].head()

,promo_dependency_score,discount_sensitivity_segment
0,1.0,highly promo sensitive
1,1.0,highly promo sensitive
2,1.0,highly promo sensitive
3,1.0,highly promo sensitive
4,1.0,highly promo sensitive


In [93]:
df["discount_sensitivity_segment"].value_counts()

# no customer that is in the catogory "moderate"

discount_sensitivity_segment
organic                   2223
highly promo sensitive    1677
Name: count, dtype: int64

## C : Spending and Value Features
* avg_spend_per_visit
* spending_velocity_score
* customer_lifetime_value_proxy
* value_tier
* value_tier_numeric

### 1. avg_spend_per_visit

This feature gives average economic value generated per purchase cycle

It normalizes spending based on previous purchase counts

In [94]:
df["avg_spend_per_visit"] = (df["Purchase Amount (USD)"] / (df["Previous Purchases"] + 1))

In [95]:
df[
    [
        "Purchase Amount (USD)",
        "Previous Purchases",
        "avg_spend_per_visit"
    ]
].head()

,Purchase Amount (USD),Previous Purchases,avg_spend_per_visit
0,53,14,3.533333
1,64,2,21.333333
2,73,23,3.041667
3,90,49,1.800000
4,49,31,1.531250


### 2. spending_velocity_score

This features tells how aggresively a customer spends relative to purchase frequency

Customer who (spends more, purchase more frequently) gets higher score

In [96]:
df["spending_velocity_score"] = (df["Purchase Amount (USD)"] * df["frequency_score"])

In [97]:
df[
    [
        "Purchase Amount (USD)",
        "frequency_score",
        "spending_velocity_score"
    ]
].head()

,Purchase Amount (USD),frequency_score,spending_velocity_score
0,53,5,265
1,64,5,320
2,73,7,511
3,90,7,630
4,49,1,49


### 3. customer_lifetime_value_proxy

This feature estimates long term customer value

It used:
- Purchase Amount
- Purchase Frequency
- Previous Purchases

In [98]:
df['customer_lifetime_value_proxy'] = (df['Purchase Amount (USD)'] * df['frequency_score'] * (df['Previous Purchases'] + 1))


In [99]:
df[
    [
        "Purchase Amount (USD)",
        "Previous Purchases",
        "frequency_score",
        "customer_lifetime_value_proxy"
    ]
].head()

,Purchase Amount (USD),Previous Purchases,frequency_score,customer_lifetime_value_proxy
0,53,14,5,3975
1,64,2,5,960
2,73,23,7,12264
3,90,49,7,31500
4,49,31,1,1568


### 4. value_tier

This feature segments customer into economic value catagories 

In [100]:
df["value_tier"] = pd.qcut(
    df["customer_lifetime_value_proxy"],
    q=4,
    labels=[
        "low",
        "medium",
        "high",
        "premium"
    ]
)

In [101]:
df["value_tier"].value_counts()

value_tier
medium     976
low        975
premium    975
high       974
Name: count, dtype: int64

### 5. value_tier_numeric

converts value_tier into numeric values

In [102]:
value_map = {
    "low": 1,
    "medium": 2,
    "high": 3,
    "premium": 4
}

df["value_tier_numeric"] = (
    df["value_tier"]
    .map(value_map)
)

df["value_tier_numeric"] = (
    df["value_tier_numeric"]
    .astype(int)
)

In [103]:
df[
    [
        "value_tier",
        "value_tier_numeric"
    ]
].head()

,value_tier,value_tier_numeric
0,medium,2
1,low,1
2,premium,4
3,premium,4
4,low,1


In [104]:
df[
    [
        "avg_spend_per_visit",
        "spending_velocity_score",
        "customer_lifetime_value_proxy",
        "value_tier_numeric"
    ]
].describe()

,avg_spend_per_visit,spending_velocity_score,customer_lifetime_value_proxy,value_tier_numeric
count,3900.000000,3900.000000,3900.000000,3900.000000
mean,4.281208,235.611538,6234.790256,2.499744
std,5.902989,158.094003,5891.788655,1.118177
min,0.392157,20.000000,60.000000,1.000000
25%,1.422863,105.000000,1855.750000,1.750000
50%,2.243902,195.000000,4284.000000,2.000000
75%,4.279762,336.000000,8838.250000,3.250000
max,49.500000,700.000000,35343.000000,4.000000


## D : Satisfaction and Engagement Features
* satisfaction_flag
* customer_engagement_score
* retention_risk_score

### 1. customer_engagement_score

This features estimates overall customer engagement

It uses:
- Purchase Frequency
- Subscription Behaviour
- Satisafaction
- Repeat Purchase Behaviour

Higher Score = Stronger engagement in long term

Component - Weight
- normalized frequency :	35%
- subscription :  	    30%
- satisfaction :  	    20%
- repeat buyer : 	    15%

In [105]:
# Normalization of frequency_score
df["normalized_frequency_score"] = (df["frequency_score"] - df["frequency_score"].min()) / (df["frequency_score"].max() - df["frequency_score"].min())

In [106]:
df["customer_engagement_score"] = (
    0.35 * df["normalized_frequency_score"] +
    0.30 * df["subscription_flag"] +
    0.20 * df["satisfaction_flag"] +
    0.15 * df["repeat_buyer_flag"]
)

In [107]:
df[
    [
        "normalized_frequency_score",
        "subscription_flag",
        "satisfaction_flag",
        "repeat_buyer_flag",
        "customer_engagement_score"
    ]
].head()

,normalized_frequency_score,subscription_flag,satisfaction_flag,repeat_buyer_flag,customer_engagement_score
0,0.666667,1,0,1,0.683333
1,0.666667,1,0,0,0.533333
2,1.000000,1,0,1,0.800000
3,1.000000,1,0,1,0.800000
4,0.000000,1,0,1,0.450000


### 2. retention_risk_score

This estimates customer retention vulnerability

It uses:
- Promotional Dependency
- Purchase Frequency (low)
- Satisfaction (low)
- Repeat Behaviour (low)

High score means greater retention risk

Component - Weight
- promo dependency :	40%
- low frequency :	30%
- dissatisfaction :	20%
- non-repeat buyer :	10%

In [108]:
# low_frequency_risk = 1 - normalized_frequency_score

df["low_frequency_risk"] = (
    1 - df["normalized_frequency_score"]
)

In [109]:
df["retention_risk_score"] = (
    0.40 * df["promo_dependency_score"] +
    0.30 * df["low_frequency_risk"] +
    0.20 * (1 - df["satisfaction_flag"]) +
    0.10 * (1 - df["repeat_buyer_flag"])
)

In [110]:
df[
    [
        "customer_engagement_score",
        "retention_risk_score"
    ]
].describe()

,customer_engagement_score,retention_risk_score
count,3900.000000,3900.000000
mean,0.467239,0.453795
std,0.210562,0.245646
min,0.000000,0.000000
25%,0.325000,0.250000
50%,0.450000,0.450000
75%,0.625000,0.650000
max,1.000000,1.000000


# E : Loyalty Features
* loyalty_score_v1
* loyalty_score_v2

## 1. loyalty_score_v1

Behaviour based customer loyalty definition

It uses:
- Repeat Purchase Behaviour
- Low Promotional Dependency
- Customer Satisfaction

Higher score means more loyal customer

Component - Weight
- repeat buyer :	30%
- low promo dependency :	40%
- satisfaction :	30%

In [111]:
df["loyalty_score_v1"] = (
    0.30 * df["repeat_buyer_flag"] +
    0.40 * (1 - df["promo_dependency_score"]) +
    0.30 * df["satisfaction_flag"]
)

In [121]:
df["loyalty_score_v1"].describe()

count    3900.000000
mean        0.614385
std         0.267646
min         0.000000
25%         0.300000
50%         0.700000
75%         0.700000
max         1.000000
Name: loyalty_score_v1, dtype: float64

## 2. loyalty_score_v2

Revenue based customer loyalty definition

It used:
- Estimated Customer Lifetime Value
- Engagement Behaviour
- Subscription Behaviour

Higher score means valuable customer (loyal customer)

Component - Weight
- normalized CLV :	50%
- engagement score :	30%
- subscription :	20%

In [115]:
# CLV normalization
df["normalized_clv"] = (
    (df["customer_lifetime_value_proxy"] -
     df["customer_lifetime_value_proxy"].min()) /
    (
        df["customer_lifetime_value_proxy"].max() -
        df["customer_lifetime_value_proxy"].min()
    )
)

In [116]:
df["loyalty_score_v2"] = (
    0.50 * df["normalized_clv"] +
    0.30 * df["customer_engagement_score"] +
    0.20 * df["subscription_flag"]
)

In [123]:
df["loyalty_score_v2"].describe()

count    3900.000000
mean        0.281676
std         0.174946
min         0.000000
25%         0.139932
50%         0.239245
75%         0.406639
max         1.000000
Name: loyalty_score_v2, dtype: float64

In [119]:
df[
    [
        "loyalty_score_v1",
        "loyalty_score_v2",
        "customer_lifetime_value_proxy"
    ]
].corr()

,loyalty_score_v1,loyalty_score_v2,customer_lifetime_value_proxy
loyalty_score_v1,1.000000,-0.182929,0.134277
loyalty_score_v2,-0.182929,1.000000,0.632949
customer_lifetime_value_proxy,0.134277,0.632949,1.000000


# F : Export
to 'final_dataset_with_features.csv'

In [125]:
df.to_csv("../data/cleaned/final_dataset_with_features.csv", index=False)

In [127]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 37 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   Customer ID                    3900 non-null   int64   
 1   Age                            3900 non-null   int64   
 2   Gender                         3900 non-null   object  
 3   Item Purchased                 3900 non-null   object  
 4   Category                       3900 non-null   object  
 5   Purchase Amount (USD)          3900 non-null   int64   
 6   Location                       3900 non-null   object  
 7   Size                           3900 non-null   object  
 8   Color                          3900 non-null   object  
 9   Season                         3900 non-null   object  
 10  Review Rating                  3900 non-null   float64 
 11  Subscription Status            3900 non-null   object  
 12  Shipping Type                  390